In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
from sktime.detection.clasp import ClaSPSegmentation
from sktime.detection.stray import STRAY

df = pd.read_csv('../data/iot_telemetry_data.csv',
                 names=['ts','device','co','humidity','light',
                        'lpg','motion','smoke','temp'],
                 skiprows=1)
df['ts'] = pd.to_datetime(df['ts'].astype(float), unit='s')
df = df.sort_values(['device', 'ts'])
df['motion'] = df['motion'].astype(bool)
df['light'] = df['light'].astype(bool)

device_id = 'b8:27:eb:bf:9d:51'
device_df = df[df['device'] == device_id].set_index('ts')
temp_series = device_df['temp'].iloc[:5000]
print("Ready:", temp_series.shape)

Ready: (5000,)


In [2]:
start = time.time()
clasp = ClaSPSegmentation(period_length=50, n_cps=5)
clasp.fit(temp_series)
cp_result = clasp.predict(temp_series)
clasp_time = time.time() - start

print(f"ClaSP — Time: {clasp_time:.2f}s — Change points: {cp_result.values.tolist()}")

c:\Users\ronak\AppData\Local\Programs\Python\Python312\Lib\site-packages\sktime\transformations\series\_clasp_numba.py:52: RuntimeWarning: invalid value encountered in sqrt
  movstd = np.sqrt(segSumSq / m - (segSum / m) ** 2)
c:\Users\ronak\AppData\Local\Programs\Python\Python312\Lib\site-packages\sktime\transformations\series\_clasp_numba.py:52: RuntimeWarning: invalid value encountered in sqrt
  movstd = np.sqrt(segSumSq / m - (segSum / m) ** 2)


ClaSP — Time: 22.88s — Change points: [[4526], [923], [1208], [1930]]


In [3]:
start = time.time()
stray = STRAY(alpha=0.01)
stray.fit(temp_series)
stray_result = stray.predict(temp_series)
stray_time = time.time() - start

print(f"STRAY — Time: {stray_time:.2f}s")
print(stray_result.head(10))

AttributeError: 'STRAY' object has no attribute 'predict'

In [4]:
start = time.time()
stray = STRAY(alpha=0.01)
stray_result = stray.fit_transform(temp_series)
stray_time = time.time() - start

print(f"STRAY — Time: {stray_time:.2f}s")
print(stray_result.head(10))

STRAY — Time: 1.04s
2020-07-12 00:01:38.073572874    False
2020-07-12 00:01:41.761234999    False
2020-07-12 00:01:45.448863506    False
2020-07-12 00:01:49.136686802    False
2020-07-12 00:01:52.798517942    False
2020-07-12 00:01:56.498260498    False
2020-07-12 00:02:00.184931040    False
2020-07-12 00:02:03.872619629    False
2020-07-12 00:02:07.560188293    False
2020-07-12 00:02:11.247800827    False
Name: temp, dtype: bool


In [5]:
# Count STRAY anomalies
stray_anomalies = int(stray_result.sum())

results = pd.DataFrame({
    'Algorithm': ['ClaSPSegmentation', 'STRAY'],
    'Runtime_seconds': [round(clasp_time, 2), round(stray_time, 2)],
    'Detections': [len(cp_result), stray_anomalies],
    'Detection_Type': ['Change Points', 'Anomaly Flags']
})

print(results.to_string(index=False))
results.to_csv('../data/benchmark_results.csv', index=False)
print("\nBenchmark saved!")

        Algorithm  Runtime_seconds  Detections Detection_Type
ClaSPSegmentation            22.88           4  Change Points
            STRAY             1.04           0  Anomaly Flags

Benchmark saved!
